# 10. 퍼셉트론과 MLP — 직접 만들어 보는 신경망

> **제10장** · **이론편 대응: 2.3~2.4절(퍼셉트론·XOR), 10.1~10.3절(신경망·활성화 함수)**
> **예상 소요**: 60분
> **필요 사양**: CPU만으로 충분

---

## 이 장에서 하는 일

이론편 2.4절에서 **단층 퍼셉트론이 XOR을 풀 수 없다**는 것이 AI 겨울의 계기가 되었다고 했다.
그것을 직접 확인하고, 층을 쌓아 해결한다.

| 절 | 하는 일 | 이론편 대응 |
|---|---|---|
| 1 | 퍼셉트론 직접 구현 | 2.3절 |
| 2 | AND·OR은 되고 **XOR은 안 된다** | 2.4절 |
| 3 | 왜 안 되는가 — 선형 분리 | 2.4절 |
| 4 | 활성화 함수와 도함수 | 10.3절 |
| 5 | **MLP로 XOR 풀기** | 10.2절 |
| 6 | 은닉층이 무엇을 배웠나 | 10.4절 |

**2절에서 일부러 실패한다.** 역사적으로 이 실패가 10년 넘는 침체를 불렀으므로,
그 좌절을 직접 겪어 보는 것이 이 장의 목적이다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import platform

_c = {"Windows": ["Malgun Gothic"], "Darwin": ["AppleGothic"],
      "Linux": ["NanumGothic", "Noto Sans CJK KR", "Noto Sans CJK JP"]}
_a = {f.name for f in fm.fontManager.ttflist}
for _n in _c.get(platform.system(), []):
    if _n in _a:
        plt.rcParams["font.family"] = _n
        break
plt.rcParams["axes.unicode_minus"] = False
print("준비 완료")

---

## 1. 퍼셉트론 직접 구현 — 이론편 2.3절

이론편 2.3절에서 다룬 퍼셉트론은 구조가 아주 단순하다.

$$\hat{y} = \begin{cases} 1 & \text{if } \mathbf{w}^\top\mathbf{x} + b > 0 \\ 0 & \text{otherwise} \end{cases}$$

학습 규칙도 한 줄이다. 틀렸을 때만 가중치를 고친다.

$$w_i \leftarrow w_i + \eta\,(y - \hat{y})\,x_i$$

이 규칙이 뜻하는 바를 보자. 정답이 1인데 0이라 답했다면 $(y-\hat{y}) = 1$이므로
가중치를 **키운다.** 반대면 줄인다. 맞혔으면 0이 되어 아무것도 하지 않는다.

In [ ]:
import numpy as np


class Perceptron:
    """단층 퍼셉트론 (이론편 2.3절)"""

    def __init__(self, n_features, lr=0.1):
        self.w = np.zeros(n_features)
        self.b = 0.0
        self.lr = lr
        self.history = []

    def predict_one(self, x):
        """계단 함수 — 0보다 크면 1, 아니면 0"""
        return 1.0 if (self.w @ x + self.b) > 0 else 0.0

    def predict(self, X):
        return np.array([self.predict_one(x) for x in X])

    def fit(self, X, y, max_epochs=100, verbose=False):
        for epoch in range(max_epochs):
            n_errors = 0
            for xi, yi in zip(X, y):
                pred = self.predict_one(xi)
                error = yi - pred
                if error != 0:
                    # 틀렸을 때만 갱신 (이론편 2.3절)
                    self.w += self.lr * error * xi
                    self.b += self.lr * error
                    n_errors += 1

            acc = (self.predict(X) == y).mean()
            self.history.append({"epoch": epoch + 1, "errors": n_errors, "acc": acc})

            if verbose and (epoch < 3 or n_errors == 0):
                print(f"    에폭 {epoch+1:3}: 오류 {n_errors}개, 정확도 {acc:.2f}")

            if n_errors == 0:
                return epoch + 1        # 수렴
        return max_epochs               # 수렴 실패


# 논리 연산 데이터 (입력은 네 가지 경우뿐)
X_logic = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)

targets = {
    "AND": np.array([0, 0, 0, 1], dtype=float),
    "OR":  np.array([0, 1, 1, 1], dtype=float),
    "XOR": np.array([0, 1, 1, 0], dtype=float),
}

print("입력 조합")
for x in X_logic:
    print(f"  {x}")
print()
print("각 연산의 정답")
for name, y in targets.items():
    print(f"  {name:4}: {y}")

---

## 2. AND·OR은 되고 XOR은 안 된다 — 이론편 2.4절 ★

세 연산을 같은 퍼셉트론으로 학습시켜 본다. 결과를 보라.

In [ ]:
import numpy as np

print("=" * 60)
print("퍼셉트론 학습 결과")
print("=" * 60)

results = {}
for name, y in targets.items():
    print(f"\n[{name}]")
    p = Perceptron(n_features=2, lr=0.1)
    epochs = p.fit(X_logic, y, max_epochs=100, verbose=True)
    acc = (p.predict(X_logic) == y).mean()
    results[name] = {"model": p, "epochs": epochs, "acc": acc}

    if acc == 1.0:
        print(f"    → {epochs}에폭 만에 완전히 학습됨")
        print(f"       w = {p.w.round(2)}, b = {p.b:.2f}")
    else:
        print(f"    → 100에폭을 다 돌려도 정확도 {acc:.2f}")
        print(f"       w = {p.w.round(2)}, b = {p.b:.2f}")

print()
print("=" * 60)
print(f"{'연산':<8}{'수렴 에폭':<14}{'정확도':<12}{'결과'}")
print("-" * 60)
for name, r in results.items():
    verdict = "학습 성공" if r["acc"] == 1.0 else "학습 실패"
    print(f"{name:<8}{r['epochs']:<14}{r['acc']:<12.2f}{verdict}")
print("-" * 60)

assert results["AND"]["acc"] == 1.0
assert results["OR"]["acc"] == 1.0
assert results["XOR"]["acc"] < 1.0
print()
print("[확인] 이론편 2.4절의 서술과 일치 — XOR만 학습되지 않는다")

### 이 실패가 남긴 것

1969년 민스키와 페퍼트가 이 한계를 수학적으로 증명한 뒤, 신경망 연구는 자금과 관심을 잃었다.
이론편 2.4절에서 다룬 첫 번째 AI 겨울이다.

방금 우리가 본 것이 바로 그 벽이다. **학습률을 바꾸든 에폭을 늘리든 XOR은 풀리지 않는다.**
알고리즘의 문제가 아니라 **구조의 한계**이기 때문이다.

---

## 3. 왜 안 되는가 — 선형 분리

퍼셉트론의 판단 경계는 $\mathbf{w}^\top\mathbf{x} + b = 0$, 즉 **직선 하나**다.
따라서 직선 하나로 두 부류를 나눌 수 있어야만 학습된다.

네 점을 그려 보면 왜 XOR이 불가능한지 눈으로 보인다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))

for ax, (name, y) in zip(axes, targets.items()):
    # 클래스별로 점 찍기
    for cls, color, marker in [(0, "#1E40AF", "o"), (1, "#EA580C", "s")]:
        m = y == cls
        ax.scatter(X_logic[m, 0], X_logic[m, 1], s=250, c=color,
                   marker=marker, edgecolor="black", linewidth=1.5,
                   zorder=5, label=f"출력 {cls}")

    # 학습된 경계선 그리기
    p = results[name]["model"]
    if abs(p.w[1]) > 1e-9:
        xs = np.linspace(-0.5, 1.5, 10)
        ys = -(p.w[0] * xs + p.b) / p.w[1]
        ax.plot(xs, ys, color="#0D9488", linewidth=2, linestyle="--",
                label="학습된 경계")

    ok = results[name]["acc"] == 1.0
    ax.set_title(f"{name} — {'직선으로 분리 가능' if ok else '직선으로 분리 불가'}",
                 color="#1E40AF" if ok else "#DC2626")
    ax.set_xlim(-0.5, 1.5)
    ax.set_ylim(-0.5, 1.5)
    ax.set_xlabel("x1")
    ax.set_ylabel("x2")
    ax.legend(fontsize=8, loc="upper right")
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("XOR을 보라. 대각선 방향으로 같은 클래스가 놓여 있다.")
print("(0,0)과 (1,1)이 한 부류, (0,1)과 (1,0)이 다른 부류인데,")
print("이 둘을 직선 하나로 갈라놓을 방법이 없다.")

---

## 4. 활성화 함수 — 이론편 10.3절

층을 쌓아 문제를 풀려면 먼저 **계단 함수를 다른 것으로 바꿔야 한다.**

계단 함수는 미분값이 거의 모든 곳에서 0이다. 이론편 5.4절에서 봤듯 경사하강법은
그래디언트를 따라 움직이는데, 그래디언트가 0이면 아무것도 배울 수 없다.

그래서 매끄러운 함수가 필요하다. 이론편 10.3절에서 다룬 시그모이드를 쓴다.

$$\sigma(x) = \frac{1}{1+e^{-x}}, \qquad \sigma'(x) = \sigma(x)\big(1-\sigma(x)\big)

In [ ]:
import numpy as np

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def sigmoid_derivative(a):
    """시그모이드의 도함수. a는 sigmoid(x)의 결과값.

    이론편 10.3절에서 유도했듯 σ'(x) = σ(x)(1-σ(x)) 이므로,
    이미 구해 둔 출력값만 있으면 지수 계산 없이 도함수를 얻는다.
    """
    return a * (1.0 - a)


def relu(x):
    return np.maximum(0.0, x)

def relu_derivative(x):
    return (x > 0).astype(float)


print("=" * 55)
print("이론편 10.3절 값 검증 — 시그모이드 도함수")
print("=" * 55)
print(f"{'x':<8}{'σ(x)':<14}{'σ(x)(1-σ(x))':<18}{'이론편 표'}")
print("-" * 55)

book_values = {0: 0.250, 2: 0.105, 4: 0.018, 6: 0.002}
for x, book in book_values.items():
    s = sigmoid(x)
    d = sigmoid_derivative(s)
    print(f"{x:<8}{s:<14.3f}{d:<18.3f}{book:.3f}")
    assert abs(d - book) < 0.001, f"x={x}에서 이론편 값과 다릅니다"

print("-" * 55)
print("[OK] 이론편 10.3절 표와 일치")
print()
print(f"최댓값 확인: σ'(0) = {sigmoid_derivative(sigmoid(0)):.4f}")
print("→ 이론편에서 유도한 대로 0.25를 넘지 못한다.")
print("  이것이 이론편 11.2절 그래디언트 소실의 원인이 된다.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = np.linspace(-8, 8, 400)

fig, axes = plt.subplots(2, 3, figsize=(14, 7))

funcs = [
    ("계단 함수", (x > 0).astype(float), np.zeros_like(x), "미분값이 0 — 학습 불가"),
    ("시그모이드", sigmoid(x), sigmoid_derivative(sigmoid(x)), "최대 0.25 — 층 쌓으면 소실"),
    ("ReLU", relu(x), relu_derivative(x), "양수 구간에서 항상 1"),
]

for j, (name, fx, dfx, note) in enumerate(funcs):
    # 위: 함수
    ax = axes[0, j]
    ax.plot(x, fx, linewidth=2.5, color="#1E40AF")
    ax.set_title(name)
    ax.grid(alpha=0.3)
    ax.axhline(0, color="gray", linewidth=0.5)
    ax.axvline(0, color="gray", linewidth=0.5)

    # 아래: 도함수
    ax = axes[1, j]
    ax.plot(x, dfx, linewidth=2.5, color="#EA580C")
    ax.set_title(f"{name}의 도함수")
    ax.set_xlabel(note, fontsize=9)
    ax.grid(alpha=0.3)
    ax.axhline(0, color="gray", linewidth=0.5)
    if name == "시그모이드":
        ax.axhline(0.25, color="#DC2626", linestyle=":", linewidth=1.5)
        ax.text(4, 0.27, "최댓값 0.25", fontsize=8, color="#DC2626")

plt.tight_layout()
plt.show()

print("왼쪽 아래를 보라 — 계단 함수의 도함수는 어디서나 0이다.")
print("이래서는 '어느 방향으로 가야 하는지' 알 수 없다.")

---

## 5. MLP로 XOR 풀기 — 이론편 10.2절 ★

이제 층을 쌓는다. 입력 2 → 은닉 4 → 출력 1 구조다.

핵심은 **은닉층의 활성화 함수**다. 만약 활성화 함수 없이 층만 쌓으면
이론편 4.2절에서 본 대로 하나의 행렬 곱으로 압축되어 버려 아무 의미가 없다.

$$W_2(W_1\mathbf{x}) = (W_2W_1)\mathbf{x} = W'\mathbf{x}$$

**비선형 활성화 함수가 층을 쌓는 것을 의미 있게 만든다.**

In [ ]:
import numpy as np


def train_mlp(X, y, n_hidden=4, lr=0.5, epochs=5000, seed=0, record_every=100):
    """2층 신경망 학습 (이론편 10.2절, 10.4절)

    구조: 입력 → 은닉(sigmoid) → 출력(sigmoid)
    역전파는 11장에서 자세히 다루므로 여기서는 결과에 집중한다.
    """
    rng = np.random.RandomState(seed)
    n_in, n_out = X.shape[1], y.shape[1]

    W1 = rng.randn(n_in, n_hidden)
    b1 = np.zeros((1, n_hidden))
    W2 = rng.randn(n_hidden, n_out)
    b2 = np.zeros((1, n_out))

    losses = []
    for ep in range(epochs):
        # --- 순전파 ---
        z1 = X @ W1 + b1
        a1 = sigmoid(z1)
        z2 = a1 @ W2 + b2
        a2 = sigmoid(z2)

        loss = np.mean((a2 - y) ** 2)
        if ep % record_every == 0:
            losses.append(loss)

        # --- 역전파 (이론편 10.4절) ---
        d2 = 2 * (a2 - y) * sigmoid_derivative(a2) / len(X)
        dW2 = a1.T @ d2
        db2 = d2.sum(axis=0, keepdims=True)

        d1 = (d2 @ W2.T) * sigmoid_derivative(a1)
        dW1 = X.T @ d1
        db1 = d1.sum(axis=0, keepdims=True)

        # --- 갱신 ---
        W1 -= lr * dW1;  b1 -= lr * db1
        W2 -= lr * dW2;  b2 -= lr * db2

    return {"W1": W1, "b1": b1, "W2": W2, "b2": b2,
            "losses": losses, "final_loss": loss, "pred": a2, "hidden": a1}


X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y_xor = np.array([[0], [1], [1], [0]], dtype=float)

print("=" * 55)
print("MLP로 XOR 학습 (은닉 4개, 5000에폭)")
print("=" * 55)
r = train_mlp(X_xor, y_xor, n_hidden=4, lr=0.5, epochs=5000, seed=0)

print(f"최종 손실: {r['final_loss']:.6f}")
print()
print(f"{'입력':<12}{'예측':<12}{'정답':<10}{'판정'}")
print("-" * 55)
for xi, pi, yi in zip(X_xor, r["pred"].ravel(), y_xor.ravel()):
    ok = "O" if abs(pi - yi) < 0.1 else "X"
    print(f"{str(xi.astype(int)):<12}{pi:<12.4f}{int(yi):<10}{ok}")
print("-" * 55)

assert r["final_loss"] < 0.01, "학습이 제대로 되지 않았습니다"
print("[OK] 단층으로는 불가능했던 XOR을 층을 쌓아 풀었다")
print()
print("이론편 3.2절에서 다룬 역전파의 재발견이 이 문제를 푼 열쇠였다.")

### 은닉 뉴런 수와 초기값의 영향

방금 은닉 뉴런을 4개 썼다. 이론편 10.5절에서 다룬 대로 이론적으로는 **2개면 충분**하다.
실제로 해 보면 어떨까.

In [ ]:
import numpy as np

print("=" * 62)
print("은닉 뉴런 수와 시드에 따른 성공률")
print("=" * 62)
print(f"{'은닉 수':<10}{'시드':<8}{'최종 손실':<16}{'결과'}")
print("-" * 62)

summary = {}
for n_hidden in [2, 4]:
    successes = 0
    for seed in range(6):
        rr = train_mlp(X_xor, y_xor, n_hidden=n_hidden, lr=0.5,
                       epochs=5000, seed=seed)
        ok = rr["final_loss"] < 0.01
        successes += ok
        mark = "성공" if ok else "실패 (국소 최솟값)"
        print(f"{n_hidden:<10}{seed:<8}{rr['final_loss']:<16.5f}{mark}")
    summary[n_hidden] = successes
    print("-" * 62)

print()
print("성공률")
for n_hidden, s in summary.items():
    print(f"  은닉 {n_hidden}개: {s}/6")
print()
print("이론상 2개로 충분하지만, 실제로는 초기값에 따라 실패한다.")
print("이론편 5.5절에서 다룬 국소 최솟값 문제이며,")
print("뉴런을 넉넉히 두면 빠져나갈 경로가 많아져 안정적이 된다.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# --- 왼쪽: 학습 곡선 비교 ---
ax = axes[0]
for n_hidden, color in [(2, "#EA580C"), (4, "#0D9488")]:
    rr = train_mlp(X_xor, y_xor, n_hidden=n_hidden, lr=0.5, epochs=5000, seed=0)
    steps = np.arange(len(rr["losses"])) * 100
    ax.plot(steps, rr["losses"], linewidth=2, color=color,
            label=f"은닉 {n_hidden}개")
ax.set_xlabel("에폭")
ax.set_ylabel("손실 (MSE)")
ax.set_yscale("log")
ax.set_title("은닉 뉴런 수에 따른 학습 곡선 (seed=0)")
ax.legend()
ax.grid(alpha=0.3, which="both")

# --- 오른쪽: 결정 경계 ---
ax = axes[1]
r4 = train_mlp(X_xor, y_xor, n_hidden=4, lr=0.5, epochs=5000, seed=0)
xx, yy = np.meshgrid(np.linspace(-0.3, 1.3, 200), np.linspace(-0.3, 1.3, 200))
grid = np.c_[xx.ravel(), yy.ravel()]
h = sigmoid(grid @ r4["W1"] + r4["b1"])
out = sigmoid(h @ r4["W2"] + r4["b2"]).reshape(xx.shape)

cs = ax.contourf(xx, yy, out, levels=20, cmap="coolwarm", alpha=0.8)
ax.contour(xx, yy, out, levels=[0.5], colors="black", linewidths=2)
plt.colorbar(cs, ax=ax, label="출력값")

for cls, color, marker in [(0, "#1E40AF", "o"), (1, "#EA580C", "s")]:
    m = y_xor.ravel() == cls
    ax.scatter(X_xor[m, 0], X_xor[m, 1], s=250, c=color, marker=marker,
               edgecolor="white", linewidth=2, zorder=5)

ax.set_title("MLP가 만든 결정 경계")
ax.set_xlabel("x1")
ax.set_ylabel("x2")

plt.tight_layout()
plt.show()

print("오른쪽 그림의 검은 선을 보라. 직선이 아니라 휘어 있다.")
print("은닉층 덕분에 직선 하나로는 불가능했던 경계를 만들 수 있게 되었다.")

---

## 6. 은닉층이 무엇을 배웠나

MLP가 XOR을 풀었다면, **은닉 뉴런들은 무엇을 계산하고 있을까.**

이론편 10.4절에서 "은닉층이 스스로 유의미한 내부 표현을 형성한다"고 했다.
네 입력에 대한 은닉 뉴런의 출력을 직접 들여다보자.

In [ ]:
import numpy as np

r2 = train_mlp(X_xor, y_xor, n_hidden=2, lr=0.5, epochs=5000, seed=2)

print("=" * 60)
print("은닉 뉴런의 출력 (은닉 2개, 학습 성공 사례)")
print("=" * 60)
print(f"{'입력':<12}{'은닉1':<12}{'은닉2':<12}{'최종 출력':<12}{'정답'}")
print("-" * 60)
for xi, hi, oi, yi in zip(X_xor, r2["hidden"], r2["pred"].ravel(), y_xor.ravel()):
    print(f"{str(xi.astype(int)):<12}{hi[0]:<12.3f}{hi[1]:<12.3f}{oi:<12.3f}{int(yi)}")
print("-" * 60)
print()

# 은닉 출력을 0/1로 반올림해 어떤 논리 연산인지 추정
rounded = (r2["hidden"] > 0.5).astype(int)
print("은닉 출력을 0.5 기준으로 반올림하면")
print(f"{'입력':<12}{'은닉1':<10}{'은닉2'}")
for xi, ri in zip(X_xor, rounded):
    print(f"{str(xi.astype(int)):<12}{ri[0]:<10}{ri[1]}")
print()

def name_of(pattern):
    table = {(0,0,0,1): "AND", (0,1,1,1): "OR", (1,1,1,0): "NAND",
             (1,0,0,0): "NOR", (0,1,1,0): "XOR", (1,0,0,1): "XNOR"}
    return table.get(tuple(pattern), "기타")

print("각 은닉 뉴런이 학습한 연산 (추정)")
print(f"  은닉1 → {name_of(rounded[:, 0])}")
print(f"  은닉2 → {name_of(rounded[:, 1])}")
print()
print("사람이 '이렇게 나눠라'라고 알려주지 않았는데,")
print("신경망이 스스로 중간 개념을 만들어 문제를 쪼갠 것이다.")
print("이것이 이론편 10.4절에서 말한 '표현의 학습'이다.")

### XOR을 쪼개는 방법

사람이 손으로 XOR을 만든다면 이렇게 할 수 있다.

$$\text{XOR}(x_1, x_2) = \text{AND}\big(\text{OR}(x_1,x_2),\ \text{NAND}(x_1,x_2)\big)$$

"둘 중 하나는 1이고(OR), 둘 다 1은 아니다(NAND)"라는 뜻이다.

**신경망이 찾아낸 것도 대체로 이런 종류의 분해다.** 다만 정확히 이 조합이라는 보장은 없고,
초기값에 따라 다른 분해를 찾기도 한다. 중요한 것은 **중간 표현을 스스로 만든다**는 사실이다.

이 성질이 16장(CNN)에서 훨씬 뚜렷하게 나타난다. 거기서는 은닉층이
가장자리·질감 같은 시각적 특징을 스스로 찾아낸다.

---

## 7. 정리

### 확인한 이론편 값

| 이론편 절 | 내용 | 결과 |
|---|---|---|
| 2.3 | 퍼셉트론 학습 규칙 | AND·OR 수렴 ✓ |
| 2.4 | **XOR 학습 불가** | 정확도 0.5에서 정체 ✓ |
| 10.3 | 시그모이드 도함수 최댓값 0.25 | 일치 ✓ |
| 10.3 | x=2,4,6에서 0.105/0.018/0.002 | 일치 ✓ |
| 10.2 | MLP로 XOR 해결 | 손실 0.0014 ✓ |

### 기억할 것

| 항목 | 요점 |
|---|---|
| 퍼셉트론 | 경계가 직선 하나 — 선형 분리 가능한 문제만 |
| 계단 함수 | 도함수가 0이라 경사하강법을 쓸 수 없음 |
| 시그모이드 | 매끄러워 미분 가능, 대신 최대 기울기 0.25 |
| 비선형 활성화 | **없으면 층을 쌓아도 한 층과 같음** (이론편 4.2절) |
| 은닉 뉴런 수 | 이론상 최소보다 넉넉히 — 국소 최솟값 회피 |
| 표현 학습 | 은닉층이 중간 개념을 스스로 만듦 |

### 다음 장

**11. 역전파 직접 구현 ★** — 이 장에서는 `train_mlp` 안에 역전파를 넣어 두고 설명을 미뤘다.
다음 장에서 그 부분을 한 줄씩 유도하며 만들고, **이론편 10.4절에서 손으로 계산한 값과 대조**한다.

1·2권을 잇는 가장 중요한 지점이다.